# Flow Matching: Equality Constraints (Lagrangian)

Minimize $x^2 + y^2 + z^2$ subject to $2x - y + z = 3$

We use the Lagrangian method: condition on all partial derivatives being zero.

**Authors:** Victor Alves and John R. Kitchin

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
import os

# Force CPU for JAX (must be set before importing JAX)
os.environ['JAX_PLATFORMS'] = 'cpu'

import torch
import jax
import jax.numpy as jnp
from jax import jacobian, vmap
from scipy.optimize import minimize

# Import reusable utilities from local module
from generative_optimization import (
    generate_samples,
    cluster_stats,
    ConditionalFlowMatching
)

# Force CPU for PyTorch
device = torch.device('cpu')
print(f"Using device: {device}")

# Figure settings
mpl.rcParams['figure.facecolor'] = 'white'
mpl.rcParams['axes.facecolor'] = 'white'
mpl.rcParams['figure.dpi'] = 150

warnings.filterwarnings('ignore')

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Problem Setup

The Lagrangian is:
$$L(x, y, z, \lambda) = x^2 + y^2 + z^2 + \lambda(2x - y + z - 3)$$

At the optimum, all partial derivatives are zero:
- $\partial L/\partial x = 2x + 2\lambda = 0$
- $\partial L/\partial y = 2y - \lambda = 0$
- $\partial L/\partial z = 2z + \lambda = 0$
- $\partial L/\partial \lambda = 2x - y + z - 3 = 0$

In [ ]:
# Scipy solution for comparison
def objective(X):
    x, y, z = X
    return x**2 + y**2 + z**2

def eq_constraint(X):
    x, y, z = X
    return 2*x - y + z - 3

sol = minimize(objective, [1, -0.5, 0.5], 
               constraints={'type': 'eq', 'fun': eq_constraint})
print(f"Scipy solution: {sol.x}")
print(f"Objective: {sol.fun}")

In [ ]:
# Generate data with Lagrangian derivatives
def L(Y):
    """Lagrangian: L = f(x) + lambda * g(x)"""
    x, y, z, lam = Y
    f = x**2 + y**2 + z**2
    g = 2*x - y + z - 3
    return f + lam * g

# Sample [x, y, z, lambda]
bounds = [[-2, 2], [-2, 1], [-1, 1], [-2, 2]]
Y = generate_samples(bounds, n_samples=512, seed=42)

# Compute Lagrangian derivatives
dL = vmap(jacobian(L))(Y)
dL = np.array(dL)

print(f"Y shape: {Y.shape}")
print(f"dL shape: {dL.shape}")

In [ ]:
# Train: generate [x,y,z,lambda] conditioned on dL = 0
x_data = Y  # [x, y, z, lambda]
c_data = dL  # [dL/dx, dL/dy, dL/dz, dL/dlambda]

fm_lagrange = ConditionalFlowMatching(x_dim=4, c_dim=4, hidden_dim=128, n_layers=4)
losses = fm_lagrange.fit(x_data, c_data, epochs=1000, batch_size=64)

In [ ]:
# Sample by conditioning on all derivatives = 0
samples = fm_lagrange.sample(c_values=[[0.0, 0.0, 0.0, 0.0]], n_samples=500)

# Get mean solution
x_opt = samples.mean(axis=0)
print(f"Flow Matching solution:")
print(f"  x = {x_opt[0]:.4f}")
print(f"  y = {x_opt[1]:.4f}")
print(f"  z = {x_opt[2]:.4f}")
print(f"  λ = {x_opt[3]:.4f}")
print(f"\nExpected: x=1.0, y=-0.5, z=0.5")

In [ ]:
# Check constraint satisfaction
constraint_vals = 2*samples[:, 0] - samples[:, 1] + samples[:, 2] - 3
print(f"Constraint violation: {np.mean(np.abs(constraint_vals)):.6f}")